# PP-MAE — All 4 Options Comparison (Google Colab)

**Pathology-Prior Masked Autoencoder for Brain MRI Denoising**

This notebook runs all 4 PP-MAE options against architecture-matched baselines on real BraTS 2021 data.

| Round | PP-MAE Option | Baselines compared |
|-------|--------------|--------------------|
| 1 | CNN U-Net (Option 1) | DnCNN, UNet-L1, Noise2Noise, REDNet |
| 2 | ViT MAE 2D (Option 2) | VanillaMAE, SparK-CNN |
| 3 | Full Pipeline (Option 3) | MultiTaskUNet, TransUNet, UNETR, SwinUNETR, SeqPipeline |
| 4 | Swin Transformer (Option 4) | SwinIR-lite, Uformer-lite |

---
**Before running:**
1. `Runtime → Change runtime type → GPU (T4 or better)`
2. Upload your BraTS data to Google Drive (or upload directly below)

**Expected runtime:** ~2h on T4 (30 subjects, 20 epochs, all 4 rounds)

## Step 1 — GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}  ({mem:.1f} GB VRAM)')
else:
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → GPU')
    print('   The experiments will be very slow on CPU.')

print(f'PyTorch: {torch.__version__}')

## Step 2 — Install Dependencies

In [ ]:
%%capture
!pip install nibabel scikit-image matplotlib numpy torch torchvision

## Step 3 — Clone the Repository

In [ ]:
import os

REPO_DIR = '/content/AL-ML'
BRANCH   = 'claude/general-session-gviGa'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/abizbright1/AL-ML.git {REPO_DIR}

!ls {REPO_DIR}

## Step 4 — BraTS Data Setup

**Three options — run the one that matches your situation:**

- **Option A**: Data in Google Drive (recommended for large dataset)
- **Option B**: Upload zip file directly to Colab
- **Option C**: Demo mode (synthetic data — no upload needed, for testing)

In [ ]:
# ============================================================
# OPTION A — Google Drive (upload brats_datat.zip to Drive first)
# ============================================================
USE_DRIVE = True   # ← set False if using Option B or C

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # ← Change this to match where you put the zip in your Drive
    DRIVE_ZIP = '/content/drive/MyDrive/brats_datat.zip'
    
    if os.path.exists(DRIVE_ZIP):
        print(f'Found: {DRIVE_ZIP}')
        print('Extracting...')
        !mkdir -p /content/BraTS2021_data
        !unzip -q {DRIVE_ZIP} -d /content/BraTS2021_data
        # Extract any nested tars
        !for f in /content/BraTS2021_data/*.tar; do tar -xf "$f" -C /content/BraTS2021_data/ 2>/dev/null || true; done
        BRATS_ROOT = '/content/BraTS2021_data'
        print('Done.')
    else:
        print(f'❌ Not found: {DRIVE_ZIP}')
        print('   Upload brats_datat.zip to Google Drive first.')
        print('   Or set USE_DRIVE=False and use Option B or C.')

In [ ]:
# ============================================================
# OPTION B — Upload zip directly to Colab session
#            (file is temporary — lost when session ends)
# ============================================================
USE_UPLOAD = False   # ← set True to upload

if USE_UPLOAD:
    from google.colab import files
    print('Select your brats_datat.zip file...')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !mkdir -p /content/BraTS2021_data
    !unzip -q {zip_name} -d /content/BraTS2021_data
    !for f in /content/BraTS2021_data/*.tar; do tar -xf "$f" -C /content/BraTS2021_data/ 2>/dev/null || true; done
    BRATS_ROOT = '/content/BraTS2021_data'
    print('Upload and extraction complete.')

In [ ]:
# ============================================================
# OPTION C — Demo mode (no data needed)
#            Uses synthetic random data for quick testing.
# ============================================================
USE_DEMO = False   # ← set True to skip data entirely

if USE_DEMO:
    BRATS_ROOT = None
    print('Demo mode: synthetic data will be generated automatically.')

In [ ]:
# Verify data is accessible
if 'BRATS_ROOT' not in dir() or BRATS_ROOT is None:
    BRATS_ROOT = None
    print('⚠️  Running in demo mode (no real data). Set USE_DRIVE=True above.')
else:
    subjects = [d for d in os.listdir(BRATS_ROOT) if d.startswith('BraTS2021_')]
    # Check they are directories with NIfTI files
    subjects = [s for s in subjects if os.path.isdir(os.path.join(BRATS_ROOT, s))]
    print(f'✅ BraTS root: {BRATS_ROOT}')
    print(f'   Found {len(subjects)} subject directories')
    if subjects:
        sample = os.path.join(BRATS_ROOT, subjects[0])
        files_in_sample = os.listdir(sample)
        print(f'   Sample subject files: {files_in_sample}')

## Step 5 — Configuration

Adjust these settings before running:

In [ ]:
# ============================================================
#  EXPERIMENT CONFIGURATION
# ============================================================

# Which rounds to run: 1=CNN, 2=ViT, 3=Pipeline, 4=Swin
# Recommended starting point: '3' or '1,4' (fastest)
ROUNDS = '2,3,4'          # ← change this

# Number of subjects to load (max ~555 available)
# Colab T4: 30 subjects ≈ 1,600 slices — runs in ~2h at 20 epochs
# Colab A100: 100 subjects ≈ 5,000 slices — feasible at 30 epochs
MAX_SUBJECTS = 30         # ← change this

# Training epochs per model
EPOCHS = 20               # ← change this (10 = quick, 30 = full)

# Segmentor pre-training epochs (shared across all rounds)
SEG_EPOCHS = 20

# Spatial crop size (must be divisible by 8; 96 is standard)
PATCH_SIZE = 96

# Rician noise level added to simulate clinical scanner noise
SIGMA = 0.08

# Output directory (will be created if it doesn't exist)
# If Drive is mounted, save there so results survive session end:
if 'DRIVE_ZIP' in dir() and os.path.exists('/content/drive'):
    OUT_DIR = '/content/drive/MyDrive/PP_MAE_Results'
else:
    OUT_DIR = '/content/PP_MAE_Results'

os.makedirs(OUT_DIR, exist_ok=True)

print('Configuration:')
print(f'  Rounds       : {ROUNDS}')
print(f'  Subjects     : {MAX_SUBJECTS}')
print(f'  Epochs       : {EPOCHS}')
print(f'  Patch size   : {PATCH_SIZE}')
print(f'  Output dir   : {OUT_DIR}')
print(f'  BraTS root   : {BRATS_ROOT}')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'  Device       : {DEVICE}')

## Step 6 — Run Experiments

This single cell runs all configured rounds and saves results. Progress is printed live.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    f'{REPO_DIR}/run_all_options.py',
    '--epochs',      str(EPOCHS),
    '--seg_epochs',  str(SEG_EPOCHS),
    '--patch_size',  str(PATCH_SIZE),
    '--sigma',       str(SIGMA),
    '--max_subjects',str(MAX_SUBJECTS),
    '--rounds',      ROUNDS,
    '--out',         OUT_DIR,
]

if BRATS_ROOT is not None:
    cmd.insert(2, BRATS_ROOT)

print('Running:', ' '.join(cmd))
print('─' * 60)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
if proc.returncode == 0:
    print('\n✅ Experiments complete!')
else:
    print(f'\n❌ Process exited with code {proc.returncode}')

## Step 7 — View Results

In [ ]:
import pandas as pd
from IPython.display import display
import os

csv_path = os.path.join(OUT_DIR, 'options_results.csv')
# Also check working directory (where script saves by default)
if not os.path.exists(csv_path):
    csv_path = os.path.join(REPO_DIR, 'options_results.csv')

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print('Results table:')
    display(df.style.highlight_max(subset=df.select_dtypes('number').columns, color='#d4edda'))
else:
    print(f'CSV not found at {csv_path}')
    print('Run Step 6 first.')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image
import glob

# Show all generated plots
search_dirs = [OUT_DIR, REPO_DIR]
plot_files  = []
for d in search_dirs:
    plot_files += glob.glob(os.path.join(d, 'options_*.png'))
plot_files = sorted(set(plot_files))

if plot_files:
    for f in plot_files:
        print(f'▶ {os.path.basename(f)}')
        display(Image(f))
        print()
else:
    print('No plot files found. Run Step 6 first.')

## Step 8 — Run the 4-Option Head-to-Head Comparison

In [ ]:
# Only works if ALL 4 rounds were run (options_results.csv has all 4 PP-MAE rows)
csv_path = os.path.join(REPO_DIR, 'options_results.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.join(OUT_DIR, 'options_results.csv')

if os.path.exists(csv_path):
    result = subprocess.run(
        [sys.executable, f'{REPO_DIR}/compare_options.py', '--csv', csv_path],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('stderr:', result.stderr[:500])
    
    # Show comparison plots
    for f in glob.glob(os.path.join(REPO_DIR, 'options_compare_*.png')):
        print(f'▶ {os.path.basename(f)}')
        display(Image(f))
else:
    print('Run all 4 rounds first (ROUNDS="1,2,3,4")')

## Step 9 — Run Grading Pipeline

In [ ]:
# Runs PP-MAE denoiser → grading with RadioTransformer, CBAM-ResNet, DINOv2Probe

GRADING_SUBJECTS = MAX_SUBJECTS

cmd_grading = [
    sys.executable,
    f'{REPO_DIR}/run_grading.py',
    '--max_subjects', str(GRADING_SUBJECTS),
    '--out', OUT_DIR,
]

if BRATS_ROOT is not None:
    cmd_grading.insert(2, BRATS_ROOT)

print('Running grading pipeline...')
proc_g = subprocess.Popen(
    cmd_grading,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc_g.stdout:
    print(line, end='', flush=True)
proc_g.wait()

# Show grading plots
for f in sorted(glob.glob(os.path.join(REPO_DIR, 'grading_*.png'))):
    print(f'\n▶ {os.path.basename(f)}')
    display(Image(f))

## Step 10 — Download Results to Your Computer

In [ ]:
import zipfile
from google.colab import files

# Collect all output files
output_files = (
    glob.glob(os.path.join(REPO_DIR, 'options_*.png')) +
    glob.glob(os.path.join(REPO_DIR, 'options_*.csv')) +
    glob.glob(os.path.join(REPO_DIR, 'grading_*.png')) +
    glob.glob(os.path.join(REPO_DIR, 'grading_*.csv')) +
    glob.glob(os.path.join(OUT_DIR,  '*.png')) +
    glob.glob(os.path.join(OUT_DIR,  '*.csv'))
)
output_files = sorted(set(output_files))

if output_files:
    zip_path = '/content/PP_MAE_results.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for f in output_files:
            zf.write(f, os.path.basename(f))
    print(f'Packed {len(output_files)} files into PP_MAE_results.zip')
    files.download(zip_path)
else:
    print('No output files found. Run the experiments first (Steps 6–9).')

---
## Individual Round Cells (Optional)

Run individual rounds separately for faster iteration:

In [ ]:
# ── Round 1: CNN Family ──────────────────────────────────────────
# PP-MAE CNN vs DnCNN / UNet-L1 / Noise2Noise / REDNet

cmd_r1 = [sys.executable, f'{REPO_DIR}/run_all_options.py',
          '--epochs', str(EPOCHS), '--rounds', '1',
          '--max_subjects', str(MAX_SUBJECTS), '--out', OUT_DIR]
if BRATS_ROOT: cmd_r1.insert(2, BRATS_ROOT)

proc = subprocess.Popen(cmd_r1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# ── Round 2: ViT/MAE Family ──────────────────────────────────────
# ViT PP-MAE 2D vs VanillaMAE / SparK-CNN

cmd_r2 = [sys.executable, f'{REPO_DIR}/run_all_options.py',
          '--epochs', str(EPOCHS), '--rounds', '2',
          '--max_subjects', str(MAX_SUBJECTS), '--out', OUT_DIR]
if BRATS_ROOT: cmd_r2.insert(2, BRATS_ROOT)

proc = subprocess.Popen(cmd_r2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# ── Round 3: Multi-task / Pipeline Family ────────────────────────
# PP-MAE Pipeline vs MultiTaskUNet / TransUNet / UNETR / SwinUNETR / SeqPipeline

cmd_r3 = [sys.executable, f'{REPO_DIR}/run_all_options.py',
          '--epochs', str(EPOCHS), '--rounds', '3',
          '--max_subjects', str(MAX_SUBJECTS), '--out', OUT_DIR]
if BRATS_ROOT: cmd_r3.insert(2, BRATS_ROOT)

proc = subprocess.Popen(cmd_r3, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# ── Round 4: Swin Family ─────────────────────────────────────────
# Swin PP-MAE vs SwinIR-lite / Uformer-lite

cmd_r4 = [sys.executable, f'{REPO_DIR}/run_all_options.py',
          '--epochs', str(EPOCHS), '--rounds', '4',
          '--max_subjects', str(MAX_SUBJECTS), '--out', OUT_DIR]
if BRATS_ROOT: cmd_r4.insert(2, BRATS_ROOT)

proc = subprocess.Popen(cmd_r4, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

---
## Troubleshooting

| Error | Fix |
|-------|-----|
| `CUDA out of memory` | Reduce `MAX_SUBJECTS` or `PATCH_SIZE=64` |
| `ModuleNotFoundError` | Re-run Step 3 (clone repo) |
| `No such file or directory: brats_datat.zip` | Check Drive path in Step 4 |
| `BraTS2021_* not found` | Re-extract the zip/tar — check Step 4 |
| Runtime disconnects | Enable Drive saving in Step 4, reduce `MAX_SUBJECTS` |
| Slow training | Make sure GPU is enabled: `Runtime → Change runtime type → T4 GPU` |